In [ ]:
import learnQ as lq

## Solving for the weights using brute force approach

There are two "brute force" options:
1. Solve for the optimal weights at the final timepoint before treatment, $T$.
2. Solve for the optimal weights that achieve balance over all pre-treatment timepoints.

Let $Y_{itk}$ represent the value of outcome $k$ for unit $i$ at time $t$. Denote the vector of outcomes at time $t$ for unit $i$ by $\vec{Y}_{it}$. Denote the $(N-1,K)$ matrix of donor unit outcomes at time $t$ by $\mathbf{Y}_t$. Each column in this matrix is a vector $\vec{Y}_{it}$ for $i\in 2,\ldots,N$.
The weights for option (1) are found by solving:
$$
\argmin_{\vec{w}\in \mathbb{R}^{N-1}} ||Y_{1,T} - \mathbf{Y}_{T}\vec{w}|| \quad\text{subject to}\quad \sum_{i=1}^{N-1} w_i = 1 \quad\text{and}\quad w_i\geq 0,
$$ 
which is the quadratic program:
$$
\argmin_{w\in\mathbb{R}^{N-1}} \vec{w}^T\mathbf{Y}_T^{\top}Y_T\vec{w} + (- Y_{1T}^\top\mathbf{Y}_T)\vec{w} \quad\text{subject to}\quad \sum_{i=1}^{N-1} w_i = 1 \quad\text{and}\quad w_i\geq 0
$$
The bias of interest is simply the discrepancy at time $T+1$. Crucially, this is a poor way of constructing a synthetic control since it leverages only the information from time $T$.
Now, let $\vec{Y}_1$ be the vector of all (pre-treatment) outcomes for unit 1, consisting of the vectors $\vec{Y}_{1t}$ for $t\in 1,\ldots, T$ stacked. Similarly, let $\mathbf{Y}$ be the stacked matrices $\mathbf{Y}_t$ for $t\in 1,\ldots T$. 
The weights for option (2) are found by solving:
$$
\argmin_{w\in\mathbb{R}^{N-1}} ||Y_{1} - \mathbf{Y}\vec{w}|| \quad\text{subject to}\quad \sum_{i=1}^{N-1} w_i = 1 \quad\text{and}\quad w_i\geq 0,
$$ 
which is easily seen to be a QP as above. Though this estimate leverages all pre-treatment information, the hypothesis is that the problem of finding optimal weights is over constrained. We expect this to result in high bias of the estimate of $\vec{Y}_{1,T+1}$.


## First brute force approach:

In [ ]:
# Using the penultimate time point to fit a synthetic control

# This approach leverages the most recent data available before the intervention - it doesn't provide much of a synthetic control, since it simply uses
# the nearest observed patient to the target patient as the SC.

treated_data = pd.read_csv('treated_data.csv')
control_data = pd.read_csv('control_data.csv')

train_target_vectors, train_covariate_matrices, test_target_vector, test_covariate_matrix = morph.morph(treated_data, control_data)

t = len(train_covariate_matrices)-1 # the penultimate time point

Y_1T = train_target_vectors[t]
print("The shape of the target vector is: ",Y_1T.shape)
Y_T = train_covariate_matrices[t]
print("The shape of the covariates is: ",Y_T.shape)

N_minus_1 = Y_T.shape[1]
print("There are: ",N_minus_1," weights being estimated")
w = cp.Variable(N_minus_1, nonneg=True) # Enforces w >= 0

# Enforce the sum == 1 constraint
constraints = [cp.sum(w) == 1]

# Objective and problem definition
obj = cp.Minimize(cp.sum_squares(Y_1T - Y_T @ w))
prob = cp.Problem(obj, constraints)
prob.solve()

# Print the results
print("The estimated weights are: \n",(w.value).round(2))
print("The discrepancy is:\n",torch.sum((Y_1T - Y_T @ w.value)**2).item())
print("The estimated outcomes are:\n", (Y_T @ w.value))
print("The actual outcomes are:\n", Y_1T)

## Second Brute Force Approach

In [ ]:
# a list of tensors - each one is a vector of outcomes. There are T many of these, one for each time point.
Y_1 = train_target_vectors 
Y = train_covariate_matrices

N_minus_1 = Y[0].shape[1]

print("There are: ",N_minus_1," weights being estimated")
w = cp.Variable(N_minus_1, nonneg=True) # Enforces w >= 0

# Enforce the sum == 1 constraint
constraints = [cp.sum(w) == 1]


# Objective and problem definition
obj = cp.Minimize(sum(cp.sum_squares(target - covariates @ w) for target, covariates in zip(Y_1, Y)))
prob = cp.Problem(obj, constraints)
prob.solve()



print("The estimated weights are: \n",(w.value).round(2))
print("The discrepancy is:\n", torch.sum((test_target_vector - test_covariate_matrix @ w.value)**2).item())
print("The estimated outcomes are:\n", (test_covariate_matrix @ w.value))
print("The actual outcomes are:\n", test_target_vector)
print("The correlation coefficient is:", torch.corrcoef(torch.stack([test_target_vector, test_covariate_matrix @ w.value]))[0, 1])